# Recording results in the `passage-eval` format

A score is only reusable if it carries its context: which method, which grid, which objective, which
split, which data revision and which code. `passage-eval` stores each run as a `Submission` record:
one row of a results table plus a run folder with the metrics. This notebook scores two classical
planners, records them in a temporary results folder, and reads the table back.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It scores two planners on 32 tasks; about half a minute.

In [ ]:
resolution = 256
split = "validation"
objective = "elevation"
condition = "obstacles"
tasks = 32
output_dir = "notebook-artifacts/notebooks/passage-eval/03_leaderboard"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2259dd94e2de28ab4a56c270247a7e347ec8f133"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


from passage_eval.baselines.classical import AStarPlanner, GreedyToGoalPlanner, RandomPolicyPlanner  # noqa: E402, F401
from passage_eval.io.dataset import scene_from_row  # noqa: E402
from passage_eval.protocol.planner import Planner, PlannerOutput, evaluate_planner  # noqa: E402, F401

METRICS = ["cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
           "success_rate", "obstacle_violation_rate"]
PLANNER_COLORS = {"A*": "#238b45", "Greedy": "#1767ac", "Random walk": "#b8862b", "Straight line": "#1767ac"}


SOLVER_OF = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}


def load_tasks(resolution, objective, condition, limit):
    """First `limit` scored tasks of the split; a task without a reference route would be skipped, and in v1.0 every task has one."""
    columns = ["id", "tensor", "metadata"] + [f"path_astar_{s}_{condition}" for s in ("elevation", "slope", "energy")]
    tasks = []
    for row in source.iter_rows(resolution, split, columns=columns, samples=False):
        if row["metadata"]["paths"][SOLVER_OF[objective]][condition]["status"] != "ok":
            continue
        tasks.append(scene_from_row(row, resolution=resolution, cost_model=objective, split=split, task_variant=condition))
        if len(tasks) == limit:
            return tasks
    return tasks

## 1. Score two planners

In [ ]:
task_list = load_tasks(resolution, objective, condition, tasks)
runs = {}
for name, planner in {"A*": AStarPlanner(), "Greedy": GreedyToGoalPlanner()}.items():
    metrics, arrays = evaluate_planner(planner, task_list)
    runs[name] = (planner, metrics, arrays)
display(Markdown(f"Scored {len(task_list)} {split} tasks at {resolution}×{resolution} "
                 f"({OBJECTIVE_LABELS[objective]} objective, {condition})."))

## 2. Write the records

Each `Submission` names the method, the benchmark cell (resolution, objective, weight, condition,
split, number of tasks), the six metrics and the provenance of the data and code. `submit` appends
one row to `leaderboard.csv` and writes `metrics.json` and every exported array in a run folder:
costs, sample identities, obstacle violations, geometric errors and the scalar metric contract.
The contract comes from the evaluator's arrays. Missing, malformed or historical contracts fail;
this recording workflow does not opt into legacy scoring.

Hub data carries the full requested revision. A local dataset is labelled as local, with its match
to the declared Hub revision left unverified. The local path is recorded as supplied.
A record can also carry the host and user that produced it. Public records should not, so this
notebook sets both to `not-recorded`.


In [ ]:
import tempfile
import tomllib
from passage_eval.results import Submission, load_leaderboard, submit

code_version = tomllib.loads((project_root / "pyproject.toml").read_text())["project"]["version"] \
    if (project_root / "pyproject.toml").is_file() else "unknown"
results_root = Path(tempfile.mkdtemp(prefix="passage-results-"))
for name, (planner, metrics, arrays) in runs.items():
    contract_array = np.asarray(arrays.get("metric_contract"))
    if contract_array.ndim != 0 or contract_array.dtype.kind not in "US":
        raise ValueError("The exported metric_contract must be scalar text")
    metric_contract = str(contract_array.item())
    if metric_contract not in {"neurips-2026-submission-v1", "passage-qualified-complete-v1"}:
        raise ValueError(f"Unsupported exported metric_contract: {metric_contract!r}")
    provenance = {
        "metric_contract": metric_contract,
        "task_variant": condition,
        "dataset_repo": dataset_repo,
        "dataset_revision": dataset_revision,
        "dataset_source": "hub" if dataset_path is None else "local",
        "dataset_path": str(dataset_path) if dataset_path is not None else None,
        "dataset_revision_verified": dataset_path is None,
        "sample_ids": np.asarray(arrays["sample_id"]).tolist(),
        "scoring_api": "passage_eval.protocol.planner.evaluate_planner",
    }
    data_identity = f"{dataset_repo}@{dataset_revision}" if dataset_path is None \
        else f"local:{dataset_path} (Hub revision unverified)"
    record = Submission(
        method_name=planner.name, family="classical", architecture=name, resolution=resolution,
        cost_model=objective, cost_weight=float(task_list[0].cost_weight), split=split,
        n_samples=len(task_list), metrics={m: metrics[m] for m in metrics if m in (
            "cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
            "success_rate", "obstacle_violation_rate")},
        host="not-recorded", user="not-recorded", package_version=code_version,
        dataset_version=data_identity, extra=provenance)
    run_dir = submit(record, root=results_root, per_sample=arrays)
files = sorted(p.relative_to(results_root).as_posix() for p in results_root.rglob("*") if p.is_file())
display(Markdown("Files written in the results folder:\n" + "\n".join(f"- `{f}`" for f in files)))


## 3. Read the table back

`load_leaderboard` returns the table with one row per run. Table 1 shows the columns that identify a
run and its six metrics; the full table also holds latency fields, which classical runs leave empty.

In [ ]:
board = load_leaderboard(results_root).df
shown = board[["method_name", "family", "resolution", "cost_model", "cost_weight", "split", "n_samples", "dataset_version",
               "package_version", "cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
               "success_rate", "obstacle_violation_rate"]].copy()
shown["metric_contract"] = board["extra_json"].map(lambda value: json.loads(value)["metric_contract"])
shown["task_variant"] = board["extra_json"].map(lambda value: json.loads(value)["task_variant"])
show(shown, "Table 1. The results table read back from disk.", "{:,.4f}")
display(Markdown(f"The full table has {board.shape[1]} columns: " + ", ".join(f"`{c}`" for c in board.columns) + "."))


## Summary

In [ ]:
display(Markdown("\n".join([
    f"- Two runs were recorded with their benchmark cell, complete per-task arrays, metric contract and "
    f"code version {code_version}.",
    f"- Dataset source: `{data_identity}`; the declared Hub revision is `{dataset_revision}`.",
    "- A comparable result keeps the same cell: resolution, objective, condition and split. Results from "
    "different cells, data revisions or metric contracts do not belong in one ranking.",
    "- For a benchmark result, freeze every setting on the calibration split, then score the test split once.",
])))
report = {"resolution": resolution, "split": split, "objective": objective, "condition": condition,
          "records": shown.to_dict("records"), "provenance": board["extra_json"].map(json.loads).tolist(),
          "columns": list(board.columns)}


In [ ]:
report_file = write_report(report, output_path, f"03_leaderboard-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*